In [1]:
import os
from tqdm import tqdm
import torch
import torch.nn.functional as F
import pandas as pd
from collections import defaultdict
import numpy as np
from torch_geometric.nn import SAGEConv
from utils.data_process_train import *
from utils.data_process_test import *
from utils.evaluate_darpatc import *

/home/cs/grad/nasrm1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Helpful functions

In [2]:
class SAGENet(torch.nn.Module):
	def __init__(self, in_channels, out_channels):
		super(SAGENet, self).__init__()	
		self.conv1 = SAGEConv(in_channels, 32, normalize=False)
		self.conv2 = SAGEConv(32, out_channels, normalize=False)

	def forward(self, x, edge_index):
		x = self.conv1(x, edge_index)
		x = F.relu(x)
		x = F.dropout(x, training=self.training)
		x = self.conv2(x, edge_index)
		return F.log_softmax(x, dim=1)

In [3]:
def train(model, optimizer, device, data):
    """
    Args:
        model (nn.Module): The GNN being trained.
        data (Data): The full PyG graph object.
    Returns:
        float: Average training loss over training nodes.
    """
    model.train()
    data = data.to(device)   
    optimizer.zero_grad()

    # Forward pass over the full graph
    out = model(data.x, data.edge_index)

    # Compute loss on the training nodes only (masked by train_mask)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])

    # Backward + update
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
def final_test(mask, model, device, data, thre, fp, tn):
    """
    Args:
        data (Data): Full PyG Data object containing x, edge_index, y, etc.
        thre (float): Confidence threshold for overriding predictions.
        fp (list): List to store indices of false positives.
        tn (list): List to store indices of true negatives.
    Returns:
        float: Accuracy on the masked nodes.
    """
    model.eval()
    data = data.to(device)

    with torch.no_grad():
        # Forward pass on the full graph
        out = model(data.x, data.edge_index)   # <- pass the whole Data object
        pred = out.max(1)[1]                   # predicted labels (top-1)
        pro = F.softmax(out, dim=1)            # class probabilities
        pro1 = pro.max(1)                      # (score, class) of top prediction

        # Suppress the top prediction to compute the 2nd-best for each node
        pro_clone = pro.clone()
        for i in range(len(pro)):
            pro_clone[i][pro1[1][i]] = -1
        pro2 = pro_clone.max(1)

        # Threshold-based override: assign dummy label 100 if not confident
        for i in range(len(pro)):
            if pro1[0][i] / pro2[0][i] < thre:
                pred[i] = 100

        # Fill fp/tn lists based on mask
        for i in torch.where(mask)[0]:
            if data.y[i] != pred[i]:
                fp.append(int(i))  # false positive index
            else:
                tn.append(int(i))  # true negative index

        # Accuracy over masked nodes only
        correct = pred[mask].eq(data.y[mask]).sum().item()

    return correct / mask.sum().item()

In [5]:
def train_pro_initial(thre,data_train,feature_num,label_num):

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize model
    model = SAGENet(feature_num, label_num).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    print("Initial training")
    # Train for 15 epochs
    for epoch in range(1, 15):
        loss = train(model, optimizer, device, data_train)            
        print(f"Epoch {epoch} | Loss: {loss:.4f}")

    return model, device, data_train

# Load Dataset

In [6]:
# os.system('cp ../groundtruth/'+'fivedirections'+'.txt ../groundtruth/fivedirections/groundtruth_uuid.txt')

In [7]:
path = '../graphchi-cpp-master/graph_data/darpatc/fivedirections_train.txt'
data_train, feature_num, label_num = MyDataset(path, 'fivedirections')
dataset_train = TestDataset(data_train)
data_train = dataset_train[0] 

In [ ]:
thre = 1.0
initial_model, device, data_train_initial = train_pro_initial(thre, data_train, feature_num, label_num)


In [9]:
def train_pro_refinement(model, device, data, thre):
    """
    Returns: model (nn.Module) after refinement training.
    """
    # Define Adam optimizer for the model parameters
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    loop_num = 0        # counter for how many refinement loops/models are saved
    max_thre = 3        # max consecutive times with no TN before stopping
    bad_cnt = 0         # counter for consecutive bad iterations

    # Iterative refinement loop
    while True:
        fp, tn = [], []  
        # Evaluate the model on the test set and collect FP/TN node indices
        acc = final_test(data.test_mask, model, device, data, thre, fp, tn)

        # If no TN found, increment "bad" counter
        if len(tn) == 0:
            bad_cnt += 1
        else:
            bad_cnt = 0   # reset if TNs exist

        # Stop if too many consecutive "bad" iterations
        if bad_cnt >= max_thre:
            break

        if len(tn) > 0:
            # Remove TN nodes from both train and test masks
            for i in tn:
                data.train_mask[i] = False
                data.test_mask[i]  = False

            # Stop after saving 22 models (arbitrary safety cap)
            if loop_num >= 22:
                print("Reached 22 models, stopping refinement")
                break

            # Save current model state to disk
            torch.save(model.state_dict(), f'/home/student/nasrm1/ROOT/threaTrace/models/fivedirections/model_{loop_num}')
            loop_num += 1

            # Stop early if no FP remain (perfect separation)
            if len(fp) == 0:
                break

        # Continue refinement: retrain model on remaining nodes
        for epoch in range(1, 150):
            # Perform one epoch of training
            loss = train(model, optimizer, device, data)
            # Stop early if loss is sufficiently small
            if loss < 1:
                break

        # Print status update for this loop
        print(f"Refinement training completed for loop {loop_num} | FP: {len(fp)}, TN: {len(tn)}")

    return model, data


In [10]:
refined_model, data_train = train_pro_refinement(initial_model, device, data_train, thre)


Refinement training completed for loop 1 | FP: 79780, TN: 116009
Refinement training completed for loop 2 | FP: 11570, TN: 68210
Refinement training completed for loop 3 | FP: 3933, TN: 7637
Refinement training completed for loop 4 | FP: 2889, TN: 1044
Refinement training completed for loop 5 | FP: 2685, TN: 204
Refinement training completed for loop 6 | FP: 2452, TN: 233
Refinement training completed for loop 7 | FP: 2402, TN: 50
Refinement training completed for loop 8 | FP: 2373, TN: 29
Refinement training completed for loop 9 | FP: 2311, TN: 62
Refinement training completed for loop 10 | FP: 2236, TN: 75
Refinement training completed for loop 11 | FP: 2207, TN: 29
Refinement training completed for loop 12 | FP: 2190, TN: 17
Refinement training completed for loop 13 | FP: 2188, TN: 2
Refinement training completed for loop 13 | FP: 2188, TN: 0
Refinement training completed for loop 14 | FP: 2148, TN: 40
Refinement training completed for loop 15 | FP: 2146, TN: 2
Refinement training c

# Test: Evaluating one model

### Load test data

In [15]:
# --- Load test dataset ---
path = '../graphchi-cpp-master/graph_data/darpatc/fivedirections_test.txt' 
data1, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'fivedirections')
dataset_test = TestDatasetA(data1)
data_test = dataset_test[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

../graphchi-cpp-master/graph_data/darpatc/fivedirections_test.txt 2025-09-27 14:20:40


In [8]:
def test_one_model(mask, model, device, data, thre, nodeA):
    model.eval()
    data = data.to(device)

    # start with all nodes flagged as suspicious
    updated_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)


    # Convert malicious list to set for O(1) lookup
    malicious_set = set(nodeA)

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        pro = torch.softmax(out, dim=1)
        pro1 = pro.max(1)

        # suppress top prediction to find 2nd-best for thresholding
        pro_clone = pro.clone()
        pro_clone[torch.arange(len(pro)), pro1[1]] = -1
        pro2 = pro_clone.max(1)

        # apply threshold logic
        uncertain_mask = (pro1[0] / pro2[0]) < thre
        pred[uncertain_mask] = 100  # dummy label for uncertain predictions

        # evaluation
        TP, FP, TN, FN = 0, 0, 0, 0
        test_nodes = mask.nonzero(as_tuple=False).view(-1).tolist()

        for i in tqdm(test_nodes, desc="Evaluating nodes"):
            flagged = (pred[i] != data.y[i])  # mismatch = flagged
            
            if not flagged:
                updated_mask[i] = False   # node is cleared

            if i in malicious_set:  # malicious node
                if flagged:
                    TP += 1
                else:
                    FN += 1
            else:  # benign node
                if flagged:
                    FP += 1
                else:
                    TN += 1

        # metrics
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0

        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()

    return acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask


In [10]:
acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, initial_model, device, data_test, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

NameError: name 'initial_model' is not defined

In [11]:
acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, refined_model, device, data_test, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

NameError: name 'refined_model' is not defined

# Evaluating the model with best FP TN ratio

In [ ]:
model_good = SAGENet(feature_num, label_num).to(device)
thre = 1.0
# Load saved weights
model_good.load_state_dict(torch.load("/home/student/nasrm1/ROOT/threaTrace/models/fivedirections/model_0", map_location=device))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, model_good, device, data_test, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

Evaluating nodes: 100%|██████████| 298333/298333 [00:05<00:00, 55377.95it/s]

Test Accuracy: 0.9066
TP: 325, FP: 27551, TN: 270452, FN: 5
Precision: 0.0117, Recall: 0.9848, F1: 0.0230, FPR: 0.0925


# Evasion

# 0- build a dataframe

In [17]:
import pickle
# load ground truth and load provenance graph dataframe and output of prepare graph function
with open("/home/student/nasrm1/ROOT/threaTrace/graphchi-cpp-master/graph_data/darpatc/for_fivedirections.pkl", "rb") as file:
    df_test, nodes_test, phrases_test, labels_test, edges_test, mapp_test, indices_of_malicious_nodes, all_ids, GT_mal = pickle.load(file)


# Step 1: Benign Node Selection

In [18]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

malicious_groups = split_malicious_nodes_by_label(indices_of_malicious_nodes, labels_test)

print("The number of malicious nodes for each label is as follows:")

for label, nodes in malicious_groups.items():
    print(f"Label {label}: {len(nodes)}")  


The number of malicious nodes for each label is as follows:
Label 8: 320
Label 0: 10


In [19]:
def split_nodes_by_label(phrases, labels):

    node_groups = defaultdict(list)

    for node_idx, node_features in enumerate(phrases):
        node_label = labels[node_idx]
        node_groups[node_label].append(node_idx)

    return dict(node_groups)

all_nodes_grouped = split_nodes_by_label(phrases_test, labels_test)

print("The number of nodes (either malicious or benign) for each label is as follows:")

for label, nodes in all_nodes_grouped.items():
    print(f"Label {label}: {len(nodes)}")  


The number of nodes (either malicious or benign) for each label is as follows:
Label 8: 7363
Label 9: 283478
Label 0: 169
Label 12: 112
Label 6: 2690
Label 11: 4060
Label 2: 427
Label 10: 34


In [20]:
def split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes):
    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  

    for node_idx, node_features in enumerate(phrases):
        if node_idx not in malicious_set:
            node_label = labels[node_idx]
            node_groups[node_label].append(node_idx)

    return dict(node_groups)


all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(phrases_test, labels_test, indices_of_malicious_nodes)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes in all_nodes_grouped_excluding_malicious.items():
    print(f"Label {label}: {len(nodes)}") 


The number of candidate benign nodes for each label is as follows:
Label 8: 7043
Label 9: 283478
Label 0: 159
Label 12: 112
Label 6: 2690
Label 11: 4060
Label 2: 427
Label 10: 34


# Step 2: Minimal Interaction Selection

In [ ]:
def filter_nodes_by_phrase_length(node_groups, phrases, min_len=3, max_len=6):

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        # Apply filtering
        filtered = [idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len]
        
        # If filtering removes everything, keep original nodes
        if len(filtered) == 0:
            filtered_groups[label] = node_indices
        else:
            filtered_groups[label] = filtered

    return filtered_groups


filtered_nodes_grouped = filter_nodes_by_phrase_length(all_nodes_grouped_excluding_malicious, phrases_test)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes in filtered_nodes_grouped.items():
    print(f"Label {label}: {len(nodes)}")  


The number of candidate benign nodes for each label is as follows:
Label 8: 1400
Label 9: 30800
Label 0: 159
Label 12: 16
Label 6: 987
Label 11: 1152
Label 2: 246
Label 10: 25


# Step 3: Contextual Consistency Filtering

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):
    top_nodes_by_similarity = {}

    for label in tqdm(malicious_groups.keys()):
        if label in filtered_nodes_grouped:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Select top most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity

top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes_test)

print("The number of candidate benign nodes for each label is as follows:")

for label, top_nodes in top_nodes_by_similarity.items():
    print(f"Label {label}: {len(top_nodes)}") 


100%|██████████| 2/2 [00:00<00:00, 863.83it/s]

The number of candidate benign nodes for each label is as follows:
Label 8: 1400
Label 0: 159


# Step 4: Confidence Filtering

In [ ]:
def pick_highest_confidence_candidates(top_nodes_by_similarity, model, device, data):
    model.eval()
    data = data.to(device)

    with torch.no_grad():
        out = model(data.x, data.edge_index)  # shape: (num_nodes, num_classes)
        probs = torch.softmax(out, dim=1)     # class probabilities
        confidences, predictions = probs.max(dim=1)  # per-node confidence + predicted label

    best_candidates = {}  # {label: (node_id, confidence)}

    for label, candidate_nodes in top_nodes_by_similarity.items():
        if len(candidate_nodes) == 0:
            continue

        # Collect confidences only for candidate nodes
        candidate_confidences = confidences[candidate_nodes].cpu().numpy()

        # Pick the node with the highest confidence
        best_idx = candidate_nodes[np.argmax(candidate_confidences)]
        best_conf = candidate_confidences.max()

        best_candidates[label] = (best_idx, best_conf)

    return best_candidates

best_candidates = pick_highest_confidence_candidates(top_nodes_by_similarity, model_good, device, data_test)

for label, (node_id, conf) in best_candidates.items():
    print(f"Label {label}: Best candidate node {node_id} with confidence {conf:.4f}")


Label 8: Best candidate node 67433 with confidence 1.0000
Label 0: Best candidate node 103361 with confidence 1.0000


# Graph Structure Adjustment

In [25]:
# Step 1: Create a mapping of malicious node ID → best benign node ID (highest confidence)
malicious_to_benign = {}
for i in indices_of_malicious_nodes:
    label = labels_test[i]
    if label in best_candidates:  # Only if we found a best candidate for this label
        best_node_id, best_conf = best_candidates[label]
        malicious_to_benign[mapp_test[i]] = mapp_test[best_node_id]

print("Number of malicious nodes mapped to benign nodes:", len(malicious_to_benign))

# Step 2: Retrieve all benign node IDs that will be replaced
all_benign_replacements = set(malicious_to_benign.values())

# Step 3: Extract relevant rows for each benign ID
benign_rows_dict = {
    benign_id: df_test[(df_test['actorID'] == benign_id) | (df_test['objectID'] == benign_id)].copy()
    for benign_id in all_benign_replacements
}

new_rows = []  # Store newly modified rows

# Step 4: Replace benign IDs with malicious IDs
for malicious_id, benign_id in tqdm(malicious_to_benign.items(), desc="Processing malicious nodes"):
    if benign_id in benign_rows_dict:
        modified_rows = benign_rows_dict[benign_id].copy()  # Copy rows where benign ID appears
        # Replace benign with malicious
        modified_rows.loc[modified_rows['actorID'] == benign_id, 'actorID'] = malicious_id
        modified_rows.loc[modified_rows['objectID'] == benign_id, 'objectID'] = malicious_id
        # Append modified rows
        new_rows.append(modified_rows)

# Step 5: Concatenate once for efficiency
df_curated = pd.concat([df_test] + new_rows, ignore_index=True)

print("Number of rows in df after modification:", len(df_curated))
print("The number of triggered events are:", len(df_curated) - len(df_test))


Number of malicious nodes mapped to benign nodes: 330


Processing malicious nodes: 100%|██████████| 330/330 [00:00<00:00, 6615.49it/s]

Number of rows in df after modification: 800715
The number of triggered events are: 1770


In [ ]:
# Save the curated DataFrame to a text file (tab-separated, no header, no index)
output_path = "/home/student/nasrm1/ROOT/threaTrace/graphchi-cpp-master/graph_data/darpatc/fivedirections_test_curated.txt"
df_curated.to_csv(output_path, sep='\t', index=False, header=False)
print(f"Curated file saved as {output_path}")

Curated file saved as /home/student/nasrm1/ROOT/threaTrace/graphchi-cpp-master/graph_data/darpatc/fivedirections_test_curated.txt


# Eval

In [16]:
# --- Load test dataset ---
path = '../graphchi-cpp-master/graph_data/darpatc/fivedirections_test.txt' 
_, _, _, _, _, nodeA_test = MyDatasetA(path, 'fivedirections')

../graphchi-cpp-master/graph_data/darpatc/fivedirections_test.txt 2025-09-27 14:21:45


In [12]:
# --- Load test dataset ---
path = '../graphchi-cpp-master/graph_data/darpatc/fivedirections_test_curated.txt' 
data_curated, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'fivedirections')
dataset_curated = TestDatasetA(data_curated)
data_curated = dataset_curated[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

../graphchi-cpp-master/graph_data/darpatc/fivedirections_test_curated.txt 2025-09-27 14:19:22


In [17]:
# Path to your saved model
save_dir = "/home/student/nasrm1/ROOT/threaTrace/magdy_models/fivedirections"
save_path = os.path.join(save_dir, "best_refined_model.pkl")

# Load the whole model (architecture + weights)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_good = torch.load(save_path, map_location=device)

# Put in eval mode for inference
model_good.eval()

print("Model loaded successfully:", type(model_good))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask_evasion = test_one_model(data_curated.test_mask, model_good, device, data_curated, thre, nodeA_test)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

Model loaded successfully: <class '__main__.SAGENet'>


Evaluating nodes:   0%|          | 0/298333 [00:00<?, ?it/s]

Evaluating nodes: 100%|██████████| 298333/298333 [00:05<00:00, 55428.30it/s]

Test Accuracy: 0.8977
TP: 112, FP: 30402, TN: 267601, FN: 218
Precision: 0.0037, Recall: 0.3394, F1: 0.0073, FPR: 0.1020
